# Deep Koopman for Nonlinear Dynamics
Here we show how to build a Koopman autoencoder, since the dynamics must be linear in 
the latent space, we can use the same LRU dynamics that we use in the DSSM model. 
For the encoder and decoder, we will use MLPs.

In [ ]:

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import optax
from einops import rearrange
from jaxtyping import Array, Float, PRNGKeyArray
from lru import nu_init, theta_init
from tqdm import tqdm
from utils import (
    create_test_model,
    init_linear_weight,
    load_and_preprocess_data,
    ProgressPlotter,
    visualize_results,
)

In [ ]:
class LRUDynamics(eqx.Module):
    """
    LRU Dynamics with optional varying modulation using MLP.
    """

    # Core LRU parameters
    d_hidden: int
    r_min: float
    r_max: float
    max_phase: float
    clip_eigs: bool
    prepend_ones: bool
    use_modulation: bool

    # Learned parameters
    theta_log: Float[Array, " d_hidden"]  # Log phase parameters
    nu_log: Float[Array, " d_hidden"]  # Log radial parameters

    model: eqx.nn.MLP

    def __init__(
        self,
        model: eqx.nn.MLP,
        d_hidden: int,
        key: PRNGKeyArray,
        r_min: float = 0.99,
        r_max: float = 1.0,
        max_phase: float = 6.28,
        clip_eigs: bool = True,
        prepend_ones: bool = True,
        use_modulation: bool = False,
    ):
        self.model = model
        self.d_hidden = d_hidden
        self.r_min = r_min
        self.r_max = r_max
        self.max_phase = max_phase
        self.clip_eigs = clip_eigs
        self.prepend_ones = prepend_ones
        self.use_modulation = use_modulation

        # Initialize dynamics parameters
        key1, key2 = jax.random.split(key, 2)
        self.theta_log = theta_init(key1, (d_hidden,), max_phase)
        self.nu_log = nu_init(key2, (d_hidden,), r_min, r_max)

    def __call__(
        self,
        x: Float[Array, " d_hidden"],
        steps: int,
    ) -> Float[Array, "steps d_hidden"]:
        # Convert log parameters to eigenvalues
        A_real = -jnp.exp(self.nu_log)
        A_imag = jnp.exp(self.theta_log)

        if self.clip_eigs:
            A_real = jnp.clip(A_real, None, -1e-5)

        A_diag = jnp.exp(A_real + 1j * A_imag)
        z = jnp.repeat(A_diag[None, :], steps, axis=0)

        if self.prepend_ones:
            z = jnp.concatenate(
                [jnp.ones((1, self.d_hidden), dtype=jnp.complex64), z],
                axis=0,
            )[:-1, :]

        x_advanced = jax.lax.associative_scan(jnp.multiply, z) * x

        if self.use_modulation:
            x_magnitude = x_advanced.real**2 + x_advanced.imag**2
            modulation_real = jax.vmap(self.model)(x_magnitude)

            # Split into real/imaginary parts for complex modulation
            modulation_complex = (
                modulation_real[..., : self.d_hidden]
                + 1j * modulation_real[..., self.d_hidden :]
            )
            x_advanced = x_advanced * modulation_complex

        return x_advanced


We define 3 separate calls for the Koopman autoencoder:
encode, decode, and advance. The main call does all three in sequence, 
but we need them separate to calculate the different loss components.

In [ ]:
class KoopmanAutoencoder1D(eqx.Module):
    """1D Koopman Autoencoder with MLP encoder/decoder and LRU dynamics."""

    encoder: eqx.nn.MLP
    decoder: eqx.nn.MLP
    dynamics: LRUDynamics
    d_vars: int  # Number of variables/channels
    d_model: int  # Sequence length
    n_steps: int  # Number of prediction steps

    def __call__(self, x: Float[Array, "T W C"]) -> Float[Array, "T W C"]:
        """Forward pass through the Koopman autoencoder."""

        z = self.encode(x)  # (T, d_latent)
        z_advanced = self.advance(z[0])
        x_reconstructed = self.decode(z_advanced)

        return x_reconstructed

    def encode(self, x: Float[Array, "T W C"]) -> Float[Array, "T d_latent"]:
        """Encode sequence to latent space."""

        x_flat = rearrange(x, "t w c -> t (w c)")
        return jax.vmap(self.encoder)(x_flat)

    def decode(self, z: Float[Array, "T d_latent"]) -> Float[Array, "T W C"]:
        """Decode latent representations back to sequences."""

        z_decoded = jax.vmap(self.decoder)(z)
        return rearrange(
            z_decoded,
            "t (w c) -> t w c",
            w=self.d_model,
            c=self.d_vars,
        )

    def advance(self, z: Float[Array, " d_latent"]) -> Array:
        """Advance latent state through dynamics."""

        d_hidden_complex = z.shape[-1] // 2  # Use last dimension for latent size
        z_complex = z[:d_hidden_complex] + 1j * z[d_hidden_complex:]
        z_evolved = self.dynamics(z_complex, self.n_steps)
        return jnp.concatenate([z_evolved.real, z_evolved.imag], axis=-1)

The loss function has three components:
- Reconstruction loss: how well the autoencoder reconstructs the input sequence.
- Linear dynamics loss: how well the advanced latent state matches the encoded sequence.
- Prediction loss: how well the decoded advanced state matches the true sequence

In [ ]:
@eqx.filter_jit
def lindyn_loss(
    model: KoopmanAutoencoder1D,
    batch: Float[Array, "B T W C"],
    encdec_weight: float = 1.0,
    lindyn_weight: float = 0.1,
    pred_weight: float = 1.0,
) -> Float[Array, ""]:
    """Multi-component loss for Koopman autoencoder."""

    def single_loss(trajectory):
        # Encode all timesteps
        encoded_seq = model.encode(trajectory)  # (T, d_latent)
        decoded_seq = model.decode(encoded_seq)  # (T, W, C)

        # Advance initial state through dynamics
        z_initial = model.encode(trajectory[0:1])  # (1, d_latent)
        z_advanced = model.advance(
            z_initial[0]
        )  # Extract first timestep -> (d_latent,)

        # Get predictions by decoding advanced states
        pred = model.decode(z_advanced)  # (T, W, C)

        # Compute losses
        reconstruction_loss = jnp.mean((decoded_seq - trajectory) ** 2)
        dynamics_loss = jnp.mean((z_advanced - encoded_seq) ** 2)
        prediction_loss = jnp.mean((pred - trajectory) ** 2)

        return (
            pred_weight * prediction_loss
            + encdec_weight * reconstruction_loss
            + lindyn_weight * dynamics_loss
        )

    # Apply to batch and average
    batch_losses = jax.vmap(single_loss)(batch)
    return jnp.mean(batch_losses)

Define the training step and the factory function to create the model.

In [ ]:

@eqx.filter_jit
def training_step(
    model: KoopmanAutoencoder1D,
    optimizer,
    opt_state,
    batch: Float[Array, "B T W C"],
):
    """Single training step with gradient computation and parameter update."""

    @eqx.filter_value_and_grad
    def compute_loss(model):
        return lindyn_loss(model, batch)

    loss_value, grads = compute_loss(model)
    updates, opt_state = optimizer.update(
        grads,
        opt_state,
        model,
    )
    model = eqx.apply_updates(model, updates)

    return model, opt_state, loss_value


def train_koopman_model(
    model: KoopmanAutoencoder1D,
    train_dataloader,
    test_dataloader=None,
    n_epochs: int = 100,
    learning_rate: float = 5e-4,
    grad_clip_norm: float = 1.0,
    visualize_every_n_epochs: int = 1,
    save_plots: bool = True,
    n_steps_test: int | None = None,
    load_path: str | None = None,
):
    print("Training Koopman Autoencoder...")

    # Load pretrained model if path provided
    initial_epoch = 0
    if load_path is not None:
        from pathlib import Path

        load_path_obj = Path(load_path)
        if load_path_obj.exists():
            model = eqx.tree_deserialise_leaves(load_path, model)
            # Extract epoch number from filename pattern koopman_model_XXXXXX.eqx
            stem = load_path_obj.stem  # Gets filename without extension
            if (
                stem.startswith("koopman_model_") and len(stem) == 20
            ):  # koopman_model_ + 6 digits
                initial_epoch = int(stem[14:])  # Extract the 6-digit number
            print(
                f"Loaded pretrained model from {load_path}, "
                f"starting from epoch {initial_epoch}"
            )
        else:
            print(
                f"Warning: Load path {load_path} does not exist, starting from scratch"
            )

    # If epochs is 0, just return the loaded model without training
    if n_epochs == 0:
        # Optionally save the model even if no training occurred
        if load_path is not None:
            save_path = f"data/koopman_model_{initial_epoch:06d}.eqx"
            # Only save if it's a different path
            if str(load_path) != save_path:
                eqx.tree_serialise_leaves(save_path, model)
                print(f"Model copied to {save_path}")
        return model, np.array([]), None, initial_epoch

    # Initialize progress plotter
    plotter = None
    if save_plots and test_dataloader is not None:
        plotter = ProgressPlotter(
            output_dir="tmp_koopman",
            model_name="Koopman",
            framerate=10,
        )

    # Calculate total training steps for proper scheduling
    batches_per_epoch = len(train_dataloader)
    total_steps = n_epochs * batches_per_epoch

    # Setup optimizer with cosine schedule and gradient clipping (same as FNO)
    schedule = optax.cosine_onecycle_schedule(
        transition_steps=total_steps,
        peak_value=learning_rate,
    )
    optimizer = optax.chain(
        optax.clip_by_global_norm(grad_clip_norm),
        optax.adamw(schedule),
    )
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

    # Training loop with tqdm progress bar
    losses = np.full(n_epochs, np.nan)

    with tqdm(range(n_epochs), desc="Training Koopman") as pbar:
        for epoch in pbar:
            epoch_losses = []

            # Iterate through batches
            for batch in train_dataloader:
                model, opt_state, loss_value = training_step(
                    model,
                    optimizer,
                    opt_state,
                    batch,
                )
                epoch_losses.append(loss_value)
                break  # For debugging, remove this line for full training

            # Record average loss for this epoch
            avg_loss = jnp.mean(jnp.array(epoch_losses))
            losses[epoch] = avg_loss

            # Update progress bar
            pbar.set_postfix({"Loss": f"{avg_loss:.6f}"})

            # Visualization during training
            if plotter is not None and (epoch + 1) % visualize_every_n_epochs == 0:
                # Create test model with current trained weights for visualization
                current_test_model = (
                    create_test_model(model, n_steps_test)
                    if n_steps_test is not None
                    else model
                )
                plotter(
                    current_test_model,
                    test_dataloader,
                    losses,
                    show_plot=False,
                    show_loss_plot=False,
                )

    # Save the model after training
    total_epochs = initial_epoch + n_epochs
    save_path = f"data/koopman_model_{total_epochs:06d}.eqx"
    eqx.tree_serialise_leaves(save_path, model)
    print(f"Model saved to {save_path}")

    return model, losses, plotter, total_epochs


def create_koopman_model(
    key: PRNGKeyArray,
    d_model: int,
    d_vars: int,
    d_latent: int = 128,
    n_steps: int = 100,
    encoder_width: int = 256,
    encoder_depth: int = 3,
    decoder_width: int = 256,
    decoder_depth: int = 3,
    dynamics_mlp_width: int | None = None,
    dynamics_mlp_depth: int = 3,
    r_min: float = 0.99,
    r_max: float = 1.0,
    max_phase: float = 6.28,
    clip_eigs: bool = True,
    prepend_ones: bool = True,
    use_modulation: bool = False,
) -> KoopmanAutoencoder1D:
    """Factory function to create initialized Koopman autoencoder."""

    # Split keys for different components
    keys = jax.random.split(key, 4)

    # Create encoder: (sequence_length * channels) -> latent_dim
    encoder = eqx.nn.MLP(
        in_size=d_model * d_vars,  # Flattened input size
        out_size=d_latent,  # Latent space size (must be even for complex)
        width_size=encoder_width,
        depth=encoder_depth,
        activation=jax.nn.selu,
        key=keys[0],
    )
    encoder = init_linear_weight(
        encoder,
        # jax.nn.initializers.orthogonal(math.sqrt(2.0)),
        jax.nn.initializers.lecun_uniform(),
        keys[0],
    )

    # Create decoder: latent_dim -> (sequence_length * channels)
    decoder = eqx.nn.MLP(
        in_size=d_latent,
        out_size=d_model * d_vars,  # Reconstruct to original size
        width_size=decoder_width,
        depth=decoder_depth,
        activation=jax.nn.selu,
        key=keys[1],
    )
    decoder = init_linear_weight(
        decoder,
        # jax.nn.initializers.orthogonal(math.sqrt(2.0)),
        jax.nn.initializers.lecun_uniform(),
        keys[1],
    )

    # Create dynamics MLP
    d_hidden_complex = d_latent // 2  # Complex dimension
    if dynamics_mlp_width is None:
        dynamics_mlp_width = d_hidden_complex

    dynamics_mlp = eqx.nn.MLP(
        in_size=d_hidden_complex,
        out_size=2 * d_hidden_complex,  # Real and imaginary parts
        width_size=dynamics_mlp_width,
        depth=dynamics_mlp_depth,
        activation=jax.nn.selu,
        key=keys[2],
    )

    # Create dynamics
    dynamics = LRUDynamics(
        model=dynamics_mlp,
        d_hidden=d_hidden_complex,
        key=keys[3],
        r_min=r_min,
        r_max=r_max,
        max_phase=max_phase,
        clip_eigs=clip_eigs,
        prepend_ones=prepend_ones,
        use_modulation=use_modulation,
    )

    return KoopmanAutoencoder1D(
        encoder=encoder,
        decoder=decoder,
        dynamics=dynamics,
        d_vars=d_vars,
        d_model=d_model,
        n_steps=n_steps,
    )

Load data, create model, and train

In [ ]:

n_epochs = 0
n_steps_train = 100
n_steps_test = 200
batch_size = 25
learning_rate = 1e-3
grad_clip_norm = 1.0

# Load data using shared pipeline
(
    train_dataloader,
    val_dataloader,
    test_dataloader,
) = load_and_preprocess_data(
    "data/string_nonlin_100_Gaussian_16000Hz_1.0s.npy",
    batch_size=batch_size,
    n_steps_train=n_steps_train,
    n_steps_test=n_steps_test,
)

# Get data dimensions from first batch
train_sample_batch = next(iter(train_dataloader))
test_sample_batch = next(iter(test_dataloader))
B_train, T_train, W_train, C_train = train_sample_batch.shape
B_test, T_test, W_test, C_test = test_sample_batch.shape
print(f"Data dimensions: B={B_train}, T={T_train}, W={W_train}, C={C_train}")

print(train_sample_batch.shape)

# Create model
key = jax.random.PRNGKey(42)
model = create_koopman_model(
    key=key,
    d_model=W_train,
    d_vars=C_train,
    d_latent=128,
    n_steps=T_train,
    r_min=0.99,
    r_max=1.0,
    max_phase=np.pi,
    dynamics_mlp_depth=2,
    clip_eigs=False,
)

param_count = sum(
    x.size
    for x in jax.tree.leaves(
        eqx.filter(model, eqx.is_array),
    )
)
print(f"Model has {param_count} parameters")

Train the model. If you have a pretrained model, you can load it by setting `load_path`.

In [ ]:
trained_model, losses, plotter, total_epochs = train_koopman_model(
    model,
    train_dataloader,
    test_dataloader,
    n_epochs=n_epochs,
    learning_rate=learning_rate,
    grad_clip_norm=grad_clip_norm,
    n_steps_test=n_steps_test,
    save_plots=False,
    load_path="data/koopman_model_005000.eqx",
)

if losses.size > 0:
    print(f"Training completed! Final loss: {losses[-1]:.6f}")
else:
    print("No training performed (n_epochs=0). Using loaded model.")

Visualize results with the final trained model. 

In [ ]:

final_trained_model = eqx.tree_at(
    lambda m: m.n_steps,
    trained_model,
    T_test,
)

visualize_results(
    final_trained_model,
    test_dataloader,
    losses,
    model_name="Koopman",
    save_path=None,
    show_plot=True,
    show_loss_plot=False,
    n_steps_train=trained_model.n_steps,
)

print("Training completed!")

In [ ]:
# Generate animation from training frames
if plotter is not None:
    plotter.render_animation("koopman_training.webm")

The Koopman model learns to predict the nonlinear string dynamics for unseen ICs and
can extrapolate beyond the training horizon.
- Try training with much longer sequences (e.g. 1 second).
At what point does it break down?
- Since we have a single layer, we can visualise and manipulate the learned model. 
Try to change the frequency and decay by modifying the eigenvalues.
- There is an additional network that modulate the dynamics. Try turning it on/off 
to see its effect.